In [2]:
import torch 
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModel,AutoModelForSeq2SeqLM
bed_model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
gen_model_name="google/flan-t5-small"
bed_tokenizer=AutoTokenizer.from_pretrained(bed_model_name)
bed_model=AutoModel.from_pretrained(bed_model_name)
gen_tokenizer=AutoTokenizer.from_pretrained(gen_model_name)
gen_model=AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)
documents=[
    "For fat loss dinner,choose high protien,low fat,and moderate carbs",
    "Good chest exercises include bench press,incline dumbbell press,and cable fly",
    "Squats train the quadriceps,glutes,and core stability",
    "To gain high muscle,you need enough protien,calorie surplus,and progressive overload"
]
def encode_texts(texts):
    inputs=bed_tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs=bed_model(**inputs)
    token_embeddings=outputs.last_hidden_state
    attention_mask=inputs["attention_mask"]
    mask=attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    masked_embeddings=token_embeddings*mask

    sum_embeddings=torch.sum(masked_embeddings,dim=1)
    sum_mask=torch.clamp(mask.sum(dim=1),min=1e-9)

    sentence_embeddings=sum_embeddings/sum_mask
    sentence_embeddings=F.normalize(sentence_embeddings,p=2,dim=1)

    return sentence_embeddings
doc_embeddings=encode_texts(documents)
def retrieve(query,documents,doc_embeddings,top_k=2):
    query_embeddings=encode_texts([query])
    scores=torch.matmul(query_embeddings,doc_embeddings.T)
    results=[]
    topk_values,topk_indices=torch.topk(scores,k=top_k,dim=1)
    for i in range(top_k):
        idx=topk_indices[0][i].item()
        score=topk_values[0][i].item()
        results.append({
            "document":documents[idx],
            "score":score
        })
    return results
def build_rag_prompt(query,retrieved_docs):
    context="\n".join(
        [f"{i+1}.{item['document']}" for i,item in enumerate(retrieved_docs)]
    )
    prompt=(
        "Answer the question using the context.\n"
        "If the context is relevant,use it clearly and briefly.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:{query}\n"
        "Answer:"
    )
    return prompt
def generate_answer(prompt):
    inputs=gen_tokenizer(prompt,return_tensors="pt",truncation=True)
    with torch.no_grad():
        output_ids=gen_model.generate(
            **inputs,
            max_new_tokens=64
        )
    answer=gen_tokenizer.decode(output_ids[0],skip_special_tokens=True)
    return answer
query="What should I eat for dinner during  fat loss?"
retrieved_docs=retrieve(query,documents,doc_embeddings,top_k=2)
rag_prompt=build_rag_prompt(query,retrieved_docs)
answer=generate_answer(rag_prompt)
print("query",query)
print("\nRetrieved docs")
for i,item in enumerate(retrieved_docs):
    print(f"Rank{i+1}")
    print("documents:",item["document"])
    print("score:",round(item["score"]),4)

print("RAG prompt:\n")
print(rag_prompt)

print("\nGenereted answer:\n")
print(answer)
    

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12030.54it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 10236.58it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


query What should I eat for dinner during  fat loss?

Retrieved docs
Rank1
documents: For fat loss dinner,choose high protien,low fat,and moderate carbs
score: 1 4
Rank2
documents: To gain high muscle,you need enough protien,calorie surplus,and progressive overload
score: 0 4
RAG prompt:

Answer the question using the context.
If the context is relevant,use it clearly and briefly.

Context:
1.For fat loss dinner,choose high protien,low fat,and moderate carbs
2.To gain high muscle,you need enough protien,calorie surplus,and progressive overload

Question:What should I eat for dinner during  fat loss?
Answer:

Genereted answer:

1.


In [ ]:
"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
"google/flan-t5-small"
"For fat loss dinner,choose high protien,low fat,and moderate carbs"
"Good chest exercises include bench press,incline dumbbell press,and cable fly"
"Squats train the quadriceps,glutes,and core stability"
"To gain high muscle,you need enough protien,calorie surplus,and progressive overload"
"Answer the question using the context.\n"
"If the context is relevant,use it clearly and briefly.\n\n"
f"Context:\n{context}\n\n"
f"Question:{query}\n"
"Answer:"
"What should I eat for dinner during  fat loss?"